# Qwen2.5-1.5B-Instruct QLoRA Fine-Tune on nhankins/legal_contracts

This notebook fine-tunes a legal QA model using the CUAD-derived dataset on Hugging Face.
Target: `Qwen/Qwen2.5-1.5B-Instruct` on a T4 (16GB) with QLoRA.


In [1]:
# Install dependencies
!pip -q install -U transformers datasets peft bitsandbytes accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 59.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.5/527.5 kB 27.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 15.4 MB/s eta 0:00:00


In [2]:
# Check GPU
!nvidia-smi

Tue Mar 10 10:13:27 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   49C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
# Optional: Hugging Face token to avoid rate limits
import os
# os.environ['HF_TOKEN'] = 'YOUR_HF_TOKEN'

import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import BitsAndBytesConfig

model_name = 'Qwen/Qwen2.5-1.5B-Instruct'
output_dir = 'qwen2.5-1.5b-legal-qlora'
max_len = 1024  # T4-friendly
num_train_epochs = 1
train_limit = None  # set to int for quick runs (e.g., 2000)


In [ ]:
# Reuse fine-tuned weights if available
from pathlib import Path
reuse_choice = input("Reuse existing fine-tuned weights if found? [y/N]: ").strip().lower()
reuse_if_available = reuse_choice in {"y", "yes"}
output_path = Path(output_dir)
weight_files = [
    "adapter_model.safetensors", "adapter_model.bin", "adapter_config.json",
    "pytorch_model.bin", "model.safetensors"
]
weights_exist = any((output_path / f).exists() for f in weight_files)
SKIP_TRAINING = bool(reuse_if_available and weights_exist)
if SKIP_TRAINING:
    print(f"Found weights in {output_dir}. Training will be skipped.")
elif reuse_if_available:
    print("No saved weights found. Training will run.")
else:
    print("Training will run.")


In [9]:
# Build QA prompts from a script-free CUAD dataset (works with datasets>=4.x)
if SKIP_TRAINING:
    print("Skipping dataset prep (reusing weights).")
else:
    from datasets import load_dataset

    ds = load_dataset("umarbutler/better-cuad")  # parquet-backed
    raw = ds["train"]

    if train_limit:
        raw = raw.select(range(min(train_limit, len(raw))))

    answer_cols = [c for c in raw.column_names if c.endswith("-Answer")]
    text_col = "Text"
    max_questions_per_doc = 8  # keep training size sane on T4

    def build_batch(batch):
        texts = []
        for i in range(len(batch[text_col])):
            text = batch[text_col][i]
            if not text:
                continue
            count = 0
            for col in answer_cols:
                ans = batch[col][i]
                if ans is None:
                    continue
                ans = str(ans).strip()
                if not ans or ans.lower() in {"n/a", "na", "none", "no"}:
                    continue
                clause = col[:-7]  # remove "-Answer"
                prompt = (
                    "You are a legal contract assistant. Answer using only the provided context.\n\n"
                    f"Question: Highlight the parts (if any) of this contract related to {clause}.\n\n"
                    f"Context:\n{text}\n\n"
                    f"Answer:\n{ans}"
                )
                texts.append(prompt)
                count += 1
                if count >= max_questions_per_doc:
                    break
        return {"text": texts}

    qa = raw.map(build_batch, batched=True, remove_columns=raw.column_names)
    qa = qa.shuffle(seed=42)
    split = qa.train_test_split(test_size=0.05, seed=42)
    train = split["train"]
    eval_ = split["test"]


README.md:   0%|          | 0.00/812 [00:00<?, ?B/s]

cuad.jsonl:   0%|          | 0.00/32.2M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/510 [00:00<?, ? examples/s]

Map:   0%|          | 0/510 [00:00<?, ? examples/s]

In [10]:
# Tokenize + load model
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True, trust_remote_code=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=max_len)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

if SKIP_TRAINING:
    from peft import PeftModel
    model = PeftModel.from_pretrained(model, output_dir)
else:
    model = prepare_model_for_kbit_training(model)
    lora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    )
    model = get_peft_model(model, lora_config)

    train_tok = train.map(tokenize, batched=True, remove_columns=["text"])
    eval_tok = eval_.map(tokenize, batched=True, remove_columns=["text"])


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/3582 [00:00<?, ? examples/s]

Map:   0%|          | 0/189 [00:00<?, ? examples/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [12]:
# Train (compatible across transformers versions)
if SKIP_TRAINING:
    print(f"Skipping training. Using weights from {output_dir}")
else:
    from transformers import TrainingArguments

    train_args_kwargs = dict(
        output_dir=output_dir,
        per_device_train_batch_size=2,
        per_device_eval_batch_size=2,
        gradient_accumulation_steps=8,
        learning_rate=2e-4,
        num_train_epochs=num_train_epochs,
        fp16=True,
        logging_steps=50,
        eval_steps=200,
        save_steps=200,
        save_total_limit=2,
        report_to="none",
    )

    # Handle version differences
    if "eval_strategy" in TrainingArguments.__init__.__code__.co_varnames:
        train_args_kwargs["eval_strategy"] = "steps"
    else:
        train_args_kwargs["evaluation_strategy"] = "steps"

    args = TrainingArguments(**train_args_kwargs)

    data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_tok,
        eval_dataset=eval_tok,
        data_collator=data_collator,
    )

    trainer.train()
    model.save_pretrained(output_dir)
    tokenizer.save_pretrained(output_dir)


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss,Validation Loss
200,1.257165,1.233821


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


('qwen2.5-1.5b-legal-qlora/tokenizer_config.json',
 'qwen2.5-1.5b-legal-qlora/chat_template.jinja',
 'qwen2.5-1.5b-legal-qlora/tokenizer.json')

In [22]:
import torch

model.eval()
if hasattr(model, "gradient_checkpointing_disable"):
    model.gradient_checkpointing_disable()
model.config.use_cache = True

prompt = (
    "You are a legal contract assistant. Answer using only the provided context.\n\n"
    "Question: Who are the contract-holders?\n\n"
    "Context:\nThe contract-holder Name: Aya Ghaleb & Niccolo Forzano.\n\n"
    "Answer:"
)

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.inference_mode():
    output = model.generate(
        **inputs,
        max_new_tokens=32,
        do_sample=True,
        temperature=0.2,
        top_p=0.9,
        top_k=40,
        repetition_penalty=1.05,
        no_repeat_ngram_size=3,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id,
    )


gen = output[0][inputs["input_ids"].shape[1]:]
text = tokenizer.decode(gen, skip_special_tokens=True)
print(text.splitlines()[0].strip())


AYA GHALEB & NICOLÒ FORANZO
